ถ้า error หรือติดอะไร ให้ถามต่อจากแชท https://chatgpt.com/s/t_69b06c8d55088191a279e7ba12fd2e71

In [ ]:
pip install ultralytics opencv-python tensorflow numpy pandas

In [ ]:
import cv2
import numpy as np

def letterbox_resize(img, new_shape, color=(114, 114, 114)):
    # ดึงขนาดภาพเดิม (height, width)
    shape = img.shape[:2]  
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # คำนวณอัตราส่วนเพื่อย่อ/ขยาย
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

    # ขนาดใหม่ที่คงอัตราส่วนเดิม
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    
    # คำนวณพื้นที่ขอบที่ต้องเติม (Padding)
    dw = (new_shape[1] - new_unpad[0]) / 2  
    dh = (new_shape[0] - new_unpad[1]) / 2  

    # ย่อ/ขยายภาพ
    if shape[::-1] != new_unpad:  
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
        
    # เติมขอบ
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    
    return img

### **Detector**

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("models/best.pt")

def detect_cells(image):

    results = model(image)[0]

    boxes = []

    for box in results.boxes:

        x1,y1,x2,y2 = map(int,box.xyxy[0])
        conf = float(box.conf[0])

        boxes.append({
            "bbox":[x1,y1,x2,y2],
            "confidence":conf
        })

    return boxes

### **Classifier**

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("../models/classifier.h5")

CLASS_NAMES = ["normal","abnormal"]

def preprocess(img):
    # ใช้ letterbox ให้เป็นขนาด 224x224 (เติมขอบสีดำหรือสีเทา)
    img = letterbox_resize(img, new_shape=224, color=(0, 0, 0))
    img = img / 255.0
    img = np.expand_dims(img, 0)
    return img


def classify(cell_img):

    inp = preprocess(cell_img)

    pred = model.predict(inp,verbose=0)[0]

    class_id = int(np.argmax(pred))

    return {
        "class":CLASS_NAMES[class_id],
        "confidence":float(pred[class_id])
    }

### **Model pipeline**

In [ ]:
import cv2
import os
import numpy as np

from detector import detect_cells
from classifier import classify

RESULT_FOLDER = "../results"

os.makedirs(RESULT_FOLDER,exist_ok=True)


def analyze_images(images):
    cell_results = []
    image_summary = []

    for image_id, image in enumerate(images):
        
        # 1. ปรับขนาดภาพเป็น 1024 แบบ Letterbox ก่อนตรวจจับ
        image_1024 = letterbox_resize(image, new_shape=1024)

        # 2. ส่งภาพที่ปรับขนาดแล้วเข้า Detector
        boxes = detect_cells(image_1024)

        total_cells = 0
        abnormal_cells = 0
        abnormal_boxes = []

        for i, b in enumerate(boxes):
            x1, y1, x2, y2 = b["bbox"]

            # ครอปเซลล์จากภาพที่ปรับเป็น 1024 แล้ว
            crop = image_1024[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            total_cells += 1

            # ส่งภาพครอปเข้า Classifier (ซึ่งจะถูกปรับเป็น 224 ด้วย letterbox ด้านบน)
            result = classify(crop)

            if result["class"] == "abnormal":
                abnormal_cells += 1
                abnormal_boxes.append((x1, y1, x2, y2))

            cell_results.append({
                "image_id": image_id,
                "cell_id": i,
                "bbox": b["bbox"],
                "predicted_class": result["class"],
                "confidence": result["confidence"]
            })

        # วาดกล่องลงบนภาพ 1024 
        vis = image_1024.copy()

        for i, (x1, y1, x2, y2) in enumerate(abnormal_boxes):
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
            label = f"Cell {i+1}"
            cv2.putText(vis, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1, cv2.LINE_AA)

        path = f"{RESULT_FOLDER}/image_{image_id}.jpg"
        cv2.imwrite(path, vis)

        image_summary.append({
            "image_id": image_id,
            "total_cells": total_cells,
            "abnormal_cells": abnormal_cells,
            "image_path": path
        })

    return summarize(image_summary)


def summarize(image_summary):

    total_cells = sum(i["total_cells"] for i in image_summary)
    total_abnormal = sum(i["abnormal_cells"] for i in image_summary)

    suspicious = [i for i in image_summary if i["abnormal_cells"]>0]

    suspicious_sorted = sorted(
        suspicious,
        key=lambda x:x["abnormal_cells"],
        reverse=True
    )[:30]

    return {
        "total_cells_detected":total_cells,
        "total_suspicious_cells_detected":total_abnormal,
        "suspicious_images_detected":len(suspicious),
        "top_suspicious_images":suspicious_sorted
    }

### **FastAPI Backend**

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from typing import List
import numpy as np
import cv2

app = FastAPI()

# เพิ่ม CORS Middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # อนุญาตทุกโดเมน (หรือระบุ ["http://localhost:3000"] ตอนขึ้นโปรดักชัน)
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/analyze")
async def analyze(files: List[UploadFile] = File(...)):
    # โค้ดเดิม
    images = []
    for file in files:
        content = await file.read()
        arr = np.frombuffer(content, np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        images.append(img)

    result = analyze_images(images)
    return result